In [8]:
import os
os.environ["HF_HOME"] = "/workspace/hf_cache"
%cd /workspace/qwenefficientai
%pip install -q ddgs trafilatura

/workspace/qwenefficientai
Note: you may need to restart the kernel to use updated packages.


In [9]:
%pip install -q -U transformers accelerate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="cuda"
)
model.eval()

Note: you may need to restart the kernel to use updated packages.


Loading weights: 100%|██████████| 398/398 [00:01<00:00, 328.21it/s]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [10]:
import json, re
from ddgs import DDGS
import trafilatura

def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '(30.19-28.5)/30.19*100'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

def read_results_csv(query: str = "") -> str:
    """Return the contents of results/results.csv."""
    try:
        return open("results/results.csv").read()
    except FileNotFoundError:
        return "error: file not found"

TOOLS = {"calculator": calculator, "read_results_csv": read_results_csv}

tool_schemas = [
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "read_results_csv",
        "description": "Read the experiment results CSV (config name, perplexity, tokens/sec).",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": []}}},
]

def parse_tool_calls(text):
    calls = []
    for m in re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.DOTALL):
        try:
            calls.append(json.loads(m))
        except json.JSONDecodeError:
            calls.append(None)  # malformed call — worth counting later
    return calls

def web_search(query: str, max_results: int = 5) -> str:
    """Search the web, return titles/URLs/snippets."""
    try:
        results = DDGS().text(query, max_results=max_results)
        return "\n\n".join(
            f"[{i+1}] {r['title']}\n{r['href']}\n{r['body']}"
            for i, r in enumerate(results)
        ) or "no results"
    except Exception as e:
        return f"search error: {e}"

def fetch_page(url: str) -> str:
    """Fetch a web page and return its main text (truncated)."""
    try:
        html = trafilatura.fetch_url(url)
        text = trafilatura.extract(html) or "could not extract text"
        return text[:4000]  # keep context manageable
    except Exception as e:
        return f"fetch error: {e}"

TOOLS.update({"web_search": web_search, "fetch_page": fetch_page})

tool_schemas += [
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the web for current information. Returns titles, URLs, snippets.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"},
            "max_results": {"type": "integer"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "fetch_page",
        "description": "Fetch one URL from earlier search results and return its main text.",
        "parameters": {"type": "object", "properties": {
            "url": {"type": "string"}}, "required": ["url"]}}},
]

def run_agent(user_msg, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_msg}]
    for _ in range(max_turns):
        prompt = tok.apply_chat_template(
            messages, tools=tool_schemas, add_generation_prompt=True,
            tokenize=False
        )
        enc = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=2048, do_sample=False)
        reply = tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
        if verbose:
            print("── model ──\n", reply.strip(), "\n")
        calls = parse_tool_calls(reply)
        if not calls:
            return reply  # final answer
        messages.append({"role": "assistant", "content": reply})
        for call in calls:
            if call is None:
                result = "error: malformed tool call JSON"
            else:
                fn = TOOLS.get(call.get("name"))
                result = fn(**call.get("arguments", {})) if fn else "error: unknown tool"
            messages.append({"role": "tool", "content": str(result)})
    return "max turns exceeded"

In [5]:
#run_agent("Read the results CSV and tell me the baseline perplexity, "
          #"then compute what a 5% degradation from it would be.")

── model ──
 <tool_call>
{"name": "read_results_csv", "arguments": {"query": "baseline perplexity"}}
</tool_call> 

── model ──
 <tool_call>
{"name": "calculator", "arguments": {"expression": "30.186 * 0.05"}}
</tool_call> 

── model ──
 The baseline perplexity from the results is **30.186**. A 5% degradation from this value would be:

$$
30.186 \times 0.05 = 1.5093
$$

So, the degraded perplexity would be **30.186 + 1.5093 = 31.6953**. 



'The baseline perplexity from the results is **30.186**. A 5% degradation from this value would be:\n\n$$\n30.186 \\times 0.05 = 1.5093\n$$\n\nSo, the degraded perplexity would be **30.186 + 1.5093 = 31.6953**.'

In [11]:
task = """Research the person named below using web_search (and fetch_page on the most
relevant result if snippets aren't enough). Then draft a short, warm professional
email introducing me (Ethan, a student working on LLM compression) and proposing a
chat. Rules:
- Only mention facts you actually found in the search results. If you can't verify
  something specific they've done, write a good email without fabricated details.
- Reference at most 1-2 specific things, naturally, not as a list of their resume.
- Under 150 words, no subject line fluff.

Person: Song Han, MIT
"""
_ = run_agent(task, max_turns=10)

── model ──
 <tool_call>
{"name": "web_search", "arguments": {"query": "Song Han MIT research work", "max_results": 3}}
</tool_call> 

── model ──
 <tool_call>
{"name": "fetch_page", "arguments": {"url": "https://mitibm.mit.edu/people/song-han/"}}
</tool_call> 

── model ──
 Hi Song Han,

I’m Ethan, a student exploring LLM compression techniques. I’ve been following your work on deep compression and efficient inference engines—especially how pruning and sparsity improve real-world deployment. Your recent work on sparse attention in LServe and SVDQuant is particularly inspiring.  

I’d love to connect and learn more about your research. Would you be open to a brief chat?  

Best,  
Ethan 

